# 📊 Model Evaluation and Baseline Comparison
This notebook evaluates our trained Bidirectional LSTM text emotion classifier against a **TF-IDF + Naive Bayes baseline** on the held-out test dataset.


In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns


## Load Model, Tokenizer, and Test Dataset


In [ ]:
project_root = os.path.dirname(os.getcwd())
test_path = os.path.join(project_root, 'datasets', 'test.txt')
df_test = pd.read_csv(test_path, sep=';', header=None, names=['text', 'emotion'])

with open(os.path.join(project_root, 'models', 'tokenizer.pkl'), 'rb') as f:
    tokenizer = pickle.load(f)
with open(os.path.join(project_root, 'models', 'label_encoder.pkl'), 'rb') as f:
    le = pickle.load(f)

model = tf.keras.models.load_model(os.path.join(project_root, 'models', 'best_model.h5'))
print('Loaded test data shape:', df_test.shape)


## Prepare Test Sequences


In [ ]:
X_test = pad_sequences(tokenizer.texts_to_sequences(df_test['text']), maxlen=100)
y_test = le.transform(df_test['emotion'])


## Evaluate LSTM Model Performance


In [ ]:
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

lstm_acc = accuracy_score(y_test, y_pred)
print(f'LSTM Model Accuracy: {lstm_acc * 100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=le.classes_))


## Train and Evaluate Naive Bayes Baseline


In [ ]:
train_path = os.path.join(project_root, 'datasets', 'train.txt')
df_train = pd.read_csv(train_path, sep=';', header=None, names=['text', 'emotion'])

tfidf = TfidfVectorizer(max_features=10000)
X_train_tfidf = tfidf.fit_transform(df_train['text'])
X_test_tfidf = tfidf.transform(df_test['text'])

nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, le.transform(df_train['emotion']))
nb_acc = nb_model.score(X_test_tfidf, y_test)
print(f'Baseline Naive Bayes Accuracy: {nb_acc * 100:.2f}%')
print(f'LSTM Improvement over Baseline: {(lstm_acc - nb_acc)*100:.2f}%')
